## Librerias

In [1]:
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
import folium
from IPython.display import IFrame
from IPython.display import display
from shapely.geometry import LineString
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from ipywidgets import Dropdown
from ipywidgets import interact
import numpy as np
import time
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import LineString
from shapely.ops import substring

fiona.drvsupport.supported_drivers['kml'] = 'rw' # enable KML support which is disabled by default
fiona.drvsupport.supported_drivers['KML'] = 'rw' # enable KML support which is disabled by default


## Operaciones en el Mapa

In [2]:
# Abre el archivo KML y lista las capas disponibles
filename = "FORUM8 Rally Japan 2024.kml"
# Carga las diferentes layers que tiene el archivo, en este caso los diferentes Secciones de la ruta.
layers = fiona.listlayers(filename)
# Carga la layer que seleccionemos
selected_layer = layers[3]
gdf = gpd.read_file(filename, driver="KML", layer=selected_layer)
# Actualizamos la sección para medir las distancias de forma correcta
gdf = gdf.to_crs(epsg=3857)
# Convertimos el gdf a una lista en caso de que sea necesaria para plotear los diferentes segmentos de la sección
gdf_list = [gdf.iloc[[i]] for i in range(len(gdf))]
draw_section = False
# Dibujamos las secciones
if draw_section:
    fig, axs = plt.subplots(1, len(gdf_list), figsize=(10 * len(gdf_list), 10 ))
    fig.suptitle(selected_layer, fontsize=16)

    if len(gdf_list) == 1:
        gdf_list[0].plot(ax=axs, color="black", lw=4)
        axs.set_title(f"Seccion {1}, {gdf_list[0].iloc[0]['Name']}")
        axs.axis("off")
    else:
        for i, gdf_part in enumerate(gdf_list):
            gdf_list[i].plot(ax=axs[i], color="black", lw=4)
            axs[i].set_title(f"Parte {i+1}, {gdf_list[i].iloc[0]['Name']}")
            axs[i].axis("off")

# Cargamos los segmentos por separado
gdf_list_names = [gdf.iloc[i]["Name"] for i in range(len(gdf))]
selected_segment = gdf_list_names[0]
gdf_segment = gdf[gdf["Name"] == selected_segment].to_crs(epsg=3099)
gdf_segment["length_m"] = gdf_segment["geometry"].length
length_m = gdf_segment["length_m"].iloc[0]

# Dibujamos el segmento seleccionado con el mapa de fondo
# providers = list(ctx.providers.keys()) # Lista de los diferentes providers que podemos usar, para ver dentro de cada uno usar tambien .keys()
# print(providers)
draw_segment = False
if draw_segment:
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    gdf_segment.plot(ax=ax, color="black", lw=2)
    ctx.add_basemap(ax, crs=gdf_segment.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)
    ax.set_title(f"Segmento {selected_segment}")
    ax.axis("off")

## Creación de la leyenda

In [3]:
# Diccionario con Color como key y Significado como value
legend_dict = {
    'green': 'Dry',
    'blue': 'Dry + Damp Patch',
    'yellow': 'Damp',
    'orange': 'Wet',
    'red': 'Standing Water',
    'black' : "Unknown"
}
draw_legend = False
if draw_legend:

    # Crear la figura y el eje
    fig, ax = plt.subplots()

    # Añadir los rectángulos de la leyenda (2/3 de ancho) y el texto (1/3 de ancho)
    for idx, (color, label) in enumerate(legend_dict.items()):
        rect = plt.Rectangle((0, idx), 4, 1, color=color)
        ax.add_patch(rect)
        ax.text(4.5, idx + 0.5, label.upper(), verticalalignment='center', fontsize=14, horizontalalignment='left')

    ax.set_xlim(0, 4 + 2)  # rectangle width (4) plus margin
    ax.set_ylim(-.1, len(legend_dict))
    ax.axis('off')

    plt.show()

## Creación de los diferentes tramos del segment

In [9]:
def split_line_variable_lengths(gdf_segment, segment_lengths, start_distance=0, end_distance = None):
    """
    Divide una línea en segmentos de longitudes variables especificadas en un array.
    
    Parameters:
        gdf_segment (GeoDataFrame): GeoDataFrame con un único LineString en la columna 'geometry'.
        segment_lengths (list): Lista con las longitudes de cada segmento.
    
    Returns:
        GeoDataFrame: Nuevo GeoDataFrame con los segmentos generados.
    """
    # Obtener la geometría (LineString) y la longitud total
    line = gdf_segment.geometry.iloc[0]
    total_length = gdf_segment.length_m.iloc[0]

    # Verificar si la línea es válida
    if not isinstance(line, LineString):
        return gdf_segment  # Si no es una línea válida, devolvemos el mismo gdf

    # Ajustar la distancia final si no se especifica
    if end_distance is None or end_distance > total_length:
        end_distance = total_length  # Asegurarnos de no sobrepasar la longitud real de la línea

    # Lista para almacenar los segmentos resultantes
    segments = []
    current_distance = max(0, start_distance)  # Asegurar que no sea menor que 0

    # Iteramos sobre las longitudes de los segmentos
    for seg_length in segment_lengths:
        # Calculamos la distancia final del segmento
        next_distance = min(current_distance + seg_length, end_distance)  # No pasarnos del límite definido

        # Extraemos el segmento de la línea respetando la curvatura
        segment = substring(line, current_distance, next_distance)
        segments.append(segment)

        # Avanzamos la distancia inicial para el próximo segmento
        current_distance = next_distance

        # Si llegamos al final, rompemos el loop
        if current_distance >= end_distance:
            break

    # Crear un nuevo GeoDataFrame con los segmentos
    new_gdf = gpd.GeoDataFrame(geometry=segments, crs=gdf_segment.crs)

    return new_gdf

In [19]:

# Uso de la función con longitudes variables
subsegment_lengths = [1000, 2000, 1500, 2500,3000,2000,4000,3000,6000]  # Longitudes en metros
gdf_subsegments = split_line_variable_lengths(gdf_segment, subsegment_lengths, start_distance=0, end_distance=length_m)
gdf_subsegments = gdf_subsegments.geometry

draw_subsegments = False
if draw_subsegments:
    # Creamos la figura
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))

    # Definir una lista de colores (usamos un colormap de Matplotlib)
    num_segments = len(gdf_subsegments)
    colors = cm.plasma(np.linspace(0, 1, num_segments))

    # Ploteamos cada segmento asegurando que se usa el mismo `ax`
    for i, geom in enumerate(gdf_subsegments):
        x, y = geom.xy  # Extraemos las coordenadas
        ax.plot(x, y, color=colors[i], linewidth=2)  # Especificamos ax aquí

    # Ajustar relación de aspecto para que sea cuadrada
    ax.set_aspect("equal")

    # Añadir el basemap sin crear otra figura
    ctx.add_basemap(ax, crs=gdf_subsegments.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)

    # Ocultar ejes
    ax.axis("off")

    # Mostrar gráfico (asegurar que solo hay una figura activa)
    plt.show()